# AGender-MTL - Step 0: download LAT and describe it (CPU)

LAT (Kuang et al. 2022) is downloaded from PLOS (supporting file S1, 435 KB). Nothing is changed in the data;
the notebook only counts rows, labels, age groups and missing cells.

**Kaggle settings:** Accelerator = **None** (CPU). Internet = **On**.
**Inputs to add:** `agender-mtl-code`.
**Expected run time:** under a minute. Save the output as the dataset `agender-mtl-step0` for Step 1.

In [ ]:
import os
import sys
from pathlib import Path

# The code is attached as the Kaggle dataset `agender-mtl-code`. Find the `agender_mtl` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('benchmark.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-mtl-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-mtl-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender_mtl' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender_mtl`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender_mtl').exists():
        os.symlink(package.parent, code_root / 'agender_mtl')

sys.path.insert(0, str(code_root))
import agender_mtl
print('Code loaded from', Path(agender_mtl.__file__).parent)

In [ ]:
# ---- Settings ----
OUTPUT = Path('/kaggle/working/agender_mtl_outputs/step0')
TIME_BUDGET_MINUTES = 30

In [ ]:
from agender_mtl import data, environment
from agender_mtl.runner import StepRunner

DATA = OUTPUT / 'data'
DATA.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'environment': runner.run('measure the machine', environment.collect)}
summary['download'] = runner.run('download LAT from PLOS', data.fetch_lat, DATA)
print(summary['download'])
lat = runner.run('read LAT', data.read_lat, DATA / 'lat_s1.zip')
if lat is not None:
    display(lat.head())
    summary['lat'] = runner.run('describe LAT', data.describe_lat, lat)
    print(summary['lat'])
summary['steps'] = runner.steps
data.save_json(summary, OUTPUT / 'step0_summary.json')
print('Failed or skipped steps:', runner.failed() or 'none')